In [2]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pandas as pd
from io import BytesIO

# ---------------------------------------------------------
# Service Account Configuration
# ---------------------------------------------------------

# Path to your service account JSON credentials
SERVICE_ACCOUNT_FILE = r"E:\CampusX One\Data Analysis using Power BI\05_Financial Dashboard\alert-howl-501812-d1-8d1f321ad116.json"

# Google Drive API permission (Read Only)
SCOPES = [
    "https://www.googleapis.com/auth/drive.readonly"
]

# Authenticate using the service account
credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=SCOPES
)

# Build the Google Drive API service
service = build(
    "drive",
    "v3",
    credentials=credentials
)

# ---------------------------------------------------------
# Google Drive Folder ID
# ---------------------------------------------------------

FOLDER_ID = "1Prutkza1IJzeKRL4XEHX2KLjarO8yGND"

# ---------------------------------------------------------
# Function to List Files in the Folder
# ---------------------------------------------------------

def list_files(service, folder_id):
    results = service.files().list(
        q=f"'{folder_id}' in parents",
        fields="files(id, name, mimeType)"
    ).execute()

    return results.get("files", [])


# Fetch all files
files = list_files(service, FOLDER_ID)

# ---------------------------------------------------------
# Function to Download a File
# ---------------------------------------------------------

def download_drive_file(service, file_id, mime_type):
    """
    Downloads a file from Google Drive and
    returns it as a BytesIO object.
    """

    if mime_type == "application/vnd.google-apps.spreadsheet":
        # Export Google Sheets as CSV
        request = service.files().export_media(
            fileId=file_id,
            mimeType="text/csv"
        )
    else:
        # Download CSV/XLSX directly
        request = service.files().get_media(
            fileId=file_id
        )

    buffer = BytesIO()

    downloader = MediaIoBaseDownload(
        buffer,
        request
    )

    done = False

    while not done:
        status, done = downloader.next_chunk()

    buffer.seek(0)

    return buffer


# ---------------------------------------------------------
# Supported File Types
# ---------------------------------------------------------

supported_types = [
    "application/vnd.google-apps.spreadsheet",
    "text/csv",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
]

# Store all DataFrames
file_dataframes = []

# ---------------------------------------------------------
# Read Every File
# ---------------------------------------------------------

for file in files:

    file_id = file["id"]
    file_name = file["name"]
    mime_type = file["mimeType"]

    # Skip unsupported file types
    if mime_type not in supported_types:
        continue

    try:

        buffer = download_drive_file(
            service,
            file_id,
            mime_type
        )

        # Read Excel
        if mime_type == "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet":
            df = pd.read_excel(
                buffer,
                engine="openpyxl"
            )

        # Read CSV or Google Sheet
        else:
            df = pd.read_csv(buffer)

        # Optional column to identify source file
        df["source_file"] = file_name

        file_dataframes.append(df)

        print(f"Loaded: {file_name} ({len(df)} rows)")

    except Exception as e:
        print(f"Error downloading/reading {file_name}: {e}")

# ---------------------------------------------------------
# Combine All Files
# ---------------------------------------------------------

if file_dataframes:
    combined_df = pd.concat(
        file_dataframes,
        ignore_index=True
    )

    print(f"\nTotal Rows: {len(combined_df)}")
else:
    combined_df = pd.DataFrame()
    print("No supported files found.")

Loaded: combined_part_2.xlsx (6000 rows)
Loaded: combined_part_1.xlsx (6000 rows)

Total Rows: 12000


In [4]:
combined_df.head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,source_file
0,0x5c5a,CUS_0xbab0,September,Lioneld,39,758-36-8574,Scientist,119312.08,9538.088084,5,...,1200.53,37.248179,NaN,Yes,800.007565,262.22730087132254,High_spent_Small_value_payments,676.9177170131862,NaN,combined_part_2.xlsx
1,0x5c5b,CUS_0xbab0,October,Lioneld,39,758-36-8574,Scientist,119312.08,9538.088084,5,...,1200.53,26.695209,19 Years and 5 Months,Yes,800.007565,596.3739433311632,Low_spent_Medium_value_payments,362.7710745533454,NaN,combined_part_2.xlsx
2,0x5c5c,CUS_0xbab0,November,Lioneld,39,758-36-8574,Scientist,119312.08,9538.088084,5,...,1200.53,36.376074,19 Years and 6 Months,Yes,800.007565,498.0128518574914,Low_spent_Medium_value_payments,461.13216602701743,NaN,combined_part_2.xlsx
3,0x5c5d,CUS_0xbab0,December,Lioneld,39,758-36-8574,Scientist,119312.08,9538.088084,5,...,1200.53,36.298855,19 Years and 7 Months,Yes,800.007565,__10000__,Low_spent_Medium_value_payments,693.0723598485156,NaN,combined_part_2.xlsx
4,0x5c66,CUS_0xe0f,September,NaN,42,349-57-1872,Lawyer,32148.76,2571.063333,8,...,2699.66,35.398727,17 Years and 11 Months,Yes,165.111416,__10000__,!@9#%8,230.09964997473745,NaN,combined_part_2.xlsx
